In [ ]:
# Block 1: notebook description and analysis objective

#This notebook is being used to evaluate the distribution shape, tail risk, and volatility behavior of a single asset.
#Original Risk Analysis blocks included here: 6-10.


In [ ]:
# Block 2: load core libraries and instantiate helpers and plotters
# Load libraries
%load_ext autoreload
%autoreload 1
%aimport Quantapp.analytics
%aimport Quantapp.visualization
%aimport Quantapp.data
%aimport Quantapp.workflows

import logging
import warnings
import json
import os
import numpy as np
from scipy.stats import skew, kurtosis
from scipy.interpolate import PchipInterpolator
import pandas as pd
import yfinance as yf
import statsmodels.api as sm
from statsmodels.tsa.stattools import coint
from IPython.display import display
from concurrent.futures import ThreadPoolExecutor
from plotly.subplots import make_subplots
from datetime import datetime
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from Quantapp.visualization import Plotter, LineChartPlotter, CandleStickPlotter, BarChartPlotter, add_sigma_reference_lines, add_mean_reference_line, add_std_annotations, add_zone_annotation, add_horizontal_zone_trace, build_time_range_buttons, build_detail_visibility_mask, build_visibility_mask
from Quantapp.analytics import TimeSeriesAnalytics as Rolling, Helper, Models, MomentumAnalytics, RiskDistributionAnalytics, RiskRelativeAnalytics, SeriesTransforms, calculate_zscore, calculate_max_drawdown, gini_coefficient, calculate_window_metrics
from Quantapp.data import MacroDataClient, normalize_benchmark_tickers, load_benchmark_data, align_series_to_common_index
warnings.filterwarnings("ignore")
logger = logging.getLogger("yfinance")
logger.disabled = True
logger.propagate = False
rolling = Rolling()
series_transforms = SeriesTransforms()
momentum_analytics = MomentumAnalytics()
risk_relative_analytics = RiskRelativeAnalytics()
risk_distribution_analytics = RiskDistributionAnalytics()
qp = Plotter()
qe = MacroDataClient()
helper = Helper()
model = Models()
lineChartPlotter = LineChartPlotter()
candleStickPlotter = CandleStickPlotter()
barChartPlotter = BarChartPlotter()

PLOTLY_NOTEBOOK_CONFIG = {"responsive": True, "scrollZoom": True}
for renderer_name in ("plotly_mimetype", "notebook", "notebook_connected", "jupyterlab"):
    try:
        pio.renderers[renderer_name].config = PLOTLY_NOTEBOOK_CONFIG.copy()
    except Exception:
        pass

def show_plotly_figure(fig, *, config=None, **layout_kwargs):
    merged_config = PLOTLY_NOTEBOOK_CONFIG.copy()
    if config:
        merged_config.update(config)
    fig.update_layout(autosize=True, **layout_kwargs)
    fig.show(config=merged_config)


In [ ]:
# Block 3: configure analysis parameters
# Define parameters for the analysis (Adjust these as needed)
interval = '1d'
period     = '20y'
risk_free_ticker = '^IRX'  # Historical proxy used to derive the daily risk-free series
#should be a string or None
ticker_str ='mes=f'#ticker to analyze
benchmark_tickers = ['SPY']  # Benchmarks to compare against (e.g., S&P 500, Bitcoin)
length_of_plots = 20 #number of years of data to plot (after computing the period, this will be adjusted to the closest available data)
trading_strategy = 'position'  # Options: 'position', 'swing', or 'structural', # Determines the trading strategy to use for time frames
var_position_value = None  # Optional notional used to translate VaR / CVaR into dollar terms


In [ ]:
# Block 4: organize structural, position, swing timeframe lists

# Define strategy timeframe profiles
TIMEFRAME_PROFILES = {
    'swing': {'short': 3, 'mid': 9, 'long': 21},
    'position': {'short': 21, 'mid': 50, 'long': 200},
    'structural': {'short': 200, 'mid': 500, 'long': 1000},
}

strategy = str(trading_strategy).strip().lower()
if strategy not in TIMEFRAME_PROFILES:
    raise ValueError(
        f"Invalid trading_strategy '{trading_strategy}'. "
        f"Expected one of: {list(TIMEFRAME_PROFILES.keys())}"
    )

time_frame_map = TIMEFRAME_PROFILES[strategy]
time_frame_short = time_frame_map['short']
time_frame_mid = time_frame_map['mid']
time_frame_long = time_frame_map['long']

return_frequencies = ('monthly', 'weekly', 'daily')


In [ ]:
# Block 5: load and align asset, benchmark, and return data

# Download and normalize asset-level data
ticker = yf.Ticker(ticker_str).history(period=period, interval=interval)
vix = yf.Ticker('^VIX').history(period=period, interval=interval)
risk_free_proxy = yf.Ticker(risk_free_ticker).history(period=period, interval=interval)
ticker = helper.simplify_datetime_index(ticker)
vix = helper.simplify_datetime_index(vix)
risk_free_proxy = helper.simplify_datetime_index(risk_free_proxy)
if risk_free_proxy.empty or 'Close' not in risk_free_proxy:
    raise ValueError(f"No risk-free history available for {risk_free_ticker}.")
risk_free_annual_yield = risk_free_proxy['Close'].dropna().sort_index().div(100)
risk_free_daily_rate = ((1 + risk_free_annual_yield) ** (1 / 252) - 1).shift(1)

# Download benchmark data once and keep it in collections for downstream cells
benchmark_tickers = normalize_benchmark_tickers(benchmark_tickers, ticker_str)
benchmark_data, skipped_benchmarks = load_benchmark_data(benchmark_tickers, period, interval, helper)
if skipped_benchmarks:
    print(f'Skipped benchmarks with no data: {skipped_benchmarks}')

analysis_index, ticker, vix, benchmark_data = align_series_to_common_index(ticker, vix, benchmark_data)
risk_free_daily_rate = risk_free_daily_rate.reindex(ticker.index).ffill()

# Calculate asset and benchmark returns for the frequencies used elsewhere in the notebook
ticker_returns = {frequency: series_transforms.returns(ticker, frequency=frequency) for frequency in return_frequencies}
ticker_monthly_returns = ticker_returns['monthly']
ticker_weekly_returns = ticker_returns['weekly']
ticker_daily_returns = ticker_returns['daily']

benchmark_returns = {
    symbol: {frequency: series_transforms.returns(frame, frequency=frequency) for frequency in return_frequencies}
    for symbol, frame in benchmark_data.items()
}

vix_returns = {frequency: series_transforms.returns(vix, frequency=frequency) for frequency in return_frequencies}
vix_monthly_returns = vix_returns['monthly']
vix_weekly_returns = vix_returns['weekly']
vix_daily_returns = vix_returns['daily']


In [ ]:
# Block 6: analyze daily returns, skew, kurtosis, and Gini z-scores

distribution_window_options = [21, 50, 200]

distribution_context = risk_distribution_analytics.build_risk_distribution_context(
    close_series=ticker['Close'],
    windows=distribution_window_options,
    default_window=200 if 200 in distribution_window_options else max(distribution_window_options),
)

fig = lineChartPlotter.plot_distribution_shape_zscores(
    metrics_by_window=distribution_context['metrics_by_window'],
    window_options=distribution_context['windows'],
    default_window=distribution_context['default_window'],
    ticker_label=ticker_str,
    include_return_panel=False,
)
show_plotly_figure(fig)


In [ ]:
# Block 7: plot a two-sided open-to-close session trade range for long and short decisions

# Refresh the shared plotter so notebook reruns pick up local code edits without a kernel restart.
from importlib import reload
import Quantapp.visualization.line_chart_plotter as _line_chart_plotter_module
reload(_line_chart_plotter_module)
lineChartPlotter = _line_chart_plotter_module.LineChartPlotter()

trade_range_window_options = [21, 50, 200]
trade_range_default_window = 200
trade_range_interval_levels = [0.95, 0.99]
trade_range_tail_levels = [0.95, 0.99]

trade_range_history_context = risk_distribution_analytics.build_trade_range_history_context(
    price_frame=ticker[['Open', 'Close']],
    windows=trade_range_window_options,
    interval_confidence_levels=trade_range_interval_levels,
    tail_confidence_levels=trade_range_tail_levels,
    default_window=trade_range_default_window,
)

trade_range_history_fig = lineChartPlotter.plot_trade_range_history_profile(
    history_context=trade_range_history_context,
    ticker_label=ticker_str,
)
from plotly.subplots import make_subplots
import copy as _copy


def _axis_index_from_ref(axis_ref, axis_letter):
    if axis_ref in (None, axis_letter):
        return 1
    if isinstance(axis_ref, str) and axis_ref.startswith(axis_letter):
        digits = ''.join(ch for ch in axis_ref[1:] if ch.isdigit())
        return int(digits) if digits else 1
    return 1


def _shift_axis_ref(axis_ref, row_offset):
    if not isinstance(axis_ref, str) or axis_ref == 'paper':
        return axis_ref
    suffix = ' domain' if axis_ref.endswith(' domain') else ''
    base = axis_ref[:-7] if suffix else axis_ref
    if not base or base[0] not in ('x', 'y'):
        return axis_ref
    axis_letter = base[0]
    digits = base[1:]
    axis_index = int(digits) if digits else 1
    shifted_index = axis_index + row_offset
    shifted_base = axis_letter if shifted_index == 1 else f'{axis_letter}{shifted_index}'
    return shifted_base + suffix


def _stack_trade_range_figures(history_fig, cone_fig, *, title_text):
    combined_fig = make_subplots(
        rows=7,
        cols=1,
        shared_xaxes=False,
        vertical_spacing=0.04,
        row_heights=[0.16, 0.13, 0.12, 0.12, 0.14, 0.14, 0.19],
        subplot_titles=(
            'Open-to-Close Returns with Two-Sided Tail Thresholds',
            'Rolling Open-to-Close Tail Forecasts: Long Floors and Short Ceilings',
            'Rolling Open-to-Close Lower Breach Rates vs Expected',
            'Rolling Open-to-Close Upper Breach Rates vs Expected',
            'Today Open-Anchored 95% Projected Close Range (Open-to-Close)',
            'Today Open-Anchored 99% Projected Close Range (Open-to-Close)',
            'Trailing Open-to-Close Return Distribution',
        ),
    )
    subplot_title_annotations = [_copy.deepcopy(annotation).to_plotly_json() for annotation in combined_fig.layout.annotations]

    def _to_json(item):
        payload = item.to_plotly_json() if hasattr(item, 'to_plotly_json') else dict(item)
        return _copy.deepcopy(payload)

    def _shift_shapes(shapes, row_offset):
        shifted_shapes = []
        for shape in shapes:
            shape_payload = _to_json(shape)
            shape_payload['xref'] = _shift_axis_ref(shape_payload.get('xref'), row_offset)
            shape_payload['yref'] = _shift_axis_ref(shape_payload.get('yref'), row_offset)
            shifted_shapes.append(shape_payload)
        return shifted_shapes

    def _shift_annotations(annotations, row_offset):
        shifted_annotations = []
        for annotation in annotations:
            annotation_payload = _to_json(annotation)
            annotation_payload['xref'] = _shift_axis_ref(annotation_payload.get('xref'), row_offset)
            annotation_payload['yref'] = _shift_axis_ref(annotation_payload.get('yref'), row_offset)
            shifted_annotations.append(annotation_payload)
        return shifted_annotations

    def _window_title(base_title, window_label):
        suffix = '-Session Window)'
        marker = base_title.rfind('(')
        suffix_index = base_title.rfind(suffix)
        if marker != -1 and suffix_index != -1 and marker < suffix_index:
            return f"{base_title[:marker + 1]}{window_label}{suffix}"
        return base_title

    for trace in history_fig.data:
        target_row = _axis_index_from_ref(getattr(trace, 'yaxis', None), 'y')
        stacked_trace = _copy.deepcopy(trace)
        stacked_trace.xaxis = None
        stacked_trace.yaxis = None
        combined_fig.add_trace(stacked_trace, row=target_row, col=1)

    for trace in cone_fig.data:
        target_row = 4 + _axis_index_from_ref(getattr(trace, 'yaxis', None), 'y')
        stacked_trace = _copy.deepcopy(trace)
        stacked_trace.xaxis = None
        stacked_trace.yaxis = None
        combined_fig.add_trace(stacked_trace, row=target_row, col=1)

    history_static_shapes = _shift_shapes(history_fig.layout.shapes, 0)
    history_extra_annotations = _shift_annotations(history_fig.layout.annotations[4:], 0)
    cone_default_shapes = _shift_shapes(cone_fig.layout.shapes, 4)
    cone_default_annotations = _shift_annotations(cone_fig.layout.annotations[3:], 4)

    combined_fig.update_yaxes(title_text=history_fig.layout.yaxis.title.text, tickformat=history_fig.layout.yaxis.tickformat, row=1, col=1)
    combined_fig.update_yaxes(title_text=history_fig.layout.yaxis2.title.text, tickformat=history_fig.layout.yaxis2.tickformat, row=2, col=1)
    combined_fig.update_yaxes(title_text=history_fig.layout.yaxis3.title.text, tickformat=history_fig.layout.yaxis3.tickformat, row=3, col=1)
    combined_fig.update_yaxes(title_text=history_fig.layout.yaxis4.title.text, tickformat=history_fig.layout.yaxis4.tickformat, row=4, col=1)
    if history_fig.layout.xaxis4.range:
        combined_fig.update_xaxes(range=list(history_fig.layout.xaxis4.range), row=4, col=1)
    combined_fig.update_xaxes(title_text=history_fig.layout.xaxis4.title.text, row=4, col=1)

    combined_fig.update_xaxes(range=list(cone_fig.layout.xaxis.range), tickmode=cone_fig.layout.xaxis.tickmode, tickvals=list(cone_fig.layout.xaxis.tickvals), ticktext=list(cone_fig.layout.xaxis.ticktext), title_text=cone_fig.layout.xaxis.title.text, row=5, col=1)
    combined_fig.update_yaxes(title_text=cone_fig.layout.yaxis.title.text, range=list(cone_fig.layout.yaxis.range), tickprefix=cone_fig.layout.yaxis.tickprefix, row=5, col=1)
    combined_fig.update_xaxes(range=list(cone_fig.layout.xaxis2.range), tickmode=cone_fig.layout.xaxis2.tickmode, tickvals=list(cone_fig.layout.xaxis2.tickvals), ticktext=list(cone_fig.layout.xaxis2.ticktext), title_text=cone_fig.layout.xaxis2.title.text, row=6, col=1)
    combined_fig.update_yaxes(title_text=cone_fig.layout.yaxis2.title.text, range=list(cone_fig.layout.yaxis2.range), tickprefix=cone_fig.layout.yaxis2.tickprefix, row=6, col=1)
    combined_fig.update_xaxes(title_text=cone_fig.layout.xaxis3.title.text, ticksuffix=cone_fig.layout.xaxis3.ticksuffix, row=7, col=1)
    combined_fig.update_yaxes(title_text=cone_fig.layout.yaxis3.title.text, row=7, col=1)

    updatemenus = []
    history_menu = history_fig.layout.updatemenus[0] if getattr(history_fig.layout, 'updatemenus', None) else None
    cone_menu = cone_fig.layout.updatemenus[0] if getattr(cone_fig.layout, 'updatemenus', None) else None
    active_index = 0
    initial_title_text = title_text

    if history_menu is not None and cone_menu is not None:
        history_buttons = list(history_menu.buttons)
        cone_buttons = list(cone_menu.buttons)
        if len(history_buttons) == len(cone_buttons) and history_buttons:
            active_index = int(history_menu.active) if history_menu.active is not None else 0
            active_index = max(0, min(active_index, len(history_buttons) - 1))
            combined_buttons = []

            for history_button, cone_button in zip(history_buttons, cone_buttons):
                history_label = str(history_button.label)
                cone_label = str(cone_button.label)
                if history_label != cone_label:
                    raise ValueError('History and cone dropdown labels do not align for stacked trade-range figure.')

                history_args = history_button.args if history_button.args is not None else []
                cone_args = cone_button.args if cone_button.args is not None else []
                history_visibility = list(history_args[0].get('visible', [])) if history_args else []
                cone_visibility = list(cone_args[0].get('visible', [])) if cone_args else []
                if not history_visibility:
                    history_visibility = [trace.visible if trace.visible is not None else True for trace in history_fig.data]
                if not cone_visibility:
                    cone_visibility = [trace.visible if trace.visible is not None else True for trace in cone_fig.data]

                cone_layout_updates = cone_args[1] if len(cone_args) > 1 else {}
                cone_shapes = _shift_shapes(cone_layout_updates.get('shapes', cone_fig.layout.shapes), 4)
                cone_annotations = _shift_annotations(cone_layout_updates.get('annotations', cone_fig.layout.annotations)[3:], 4)
                combined_buttons.append(
                    dict(
                        label=history_label,
                        method='update',
                        args=[
                            {'visible': history_visibility + cone_visibility},
                            {
                                'title': lineChartPlotter._header_title(_window_title(title_text, history_label)),
                                'shapes': history_static_shapes + cone_shapes,
                                'annotations': subplot_title_annotations + history_extra_annotations + cone_annotations,
                            },
                        ],
                    )
                )

            initial_title_text = _window_title(title_text, str(history_buttons[active_index].label))
            updatemenus = [
                lineChartPlotter._dropdown_menu(
                    buttons=combined_buttons,
                    x=0.0,
                    active=active_index,
                )
            ]

    combined_fig.update_layout(
        title=lineChartPlotter._header_title(initial_title_text),
        height=2550,
        margin=lineChartPlotter._header_margin(top=180),
        template='plotly_dark',
        hovermode='closest',
        legend=dict(orientation='h', yanchor='bottom', y=1.01, xanchor='left', x=0),
        bargap=0.08,
        updatemenus=updatemenus,
        shapes=history_static_shapes + cone_default_shapes,
        annotations=subplot_title_annotations + history_extra_annotations + cone_default_annotations,
    )
    return combined_fig

trade_range_cone_contexts_by_window = {
    window: risk_distribution_analytics.build_trade_range_probability_context(
        price_frame=ticker[['Open', 'Close']],
        window=window,
        interval_confidence_levels=trade_range_interval_levels,
        tail_confidence_levels=trade_range_tail_levels,
    )
    for window in trade_range_window_options
}

trade_range_window = trade_range_history_context['default_window']
trade_range_context = trade_range_cone_contexts_by_window[trade_range_window]

range_summary = trade_range_context['range_summary_table'].copy()
tail_summary = trade_range_context['tail_summary_table'].copy()

range_summary = range_summary.rename(columns={
    'Confidence': 'Open-to-Close Confidence',
    'Lower Return': 'Lower Open-to-Close Return',
    'Upper Return': 'Upper Open-to-Close Return',
    'Lower Close': 'Lower Projected Close',
    'Upper Close': 'Upper Projected Close',
})
tail_summary = tail_summary.rename(columns={
    'Confidence': 'Tail Confidence',
    'Long VaR Floor': 'Open-to-Close Long VaR Floor',
    'Long CVaR Floor': 'Open-to-Close Long CVaR Floor',
    'Short VaR Ceiling': 'Open-to-Close Short VaR Ceiling',
    'Short CVaR Ceiling': 'Open-to-Close Short CVaR Ceiling',
})

if not range_summary.empty:
    for column in ('Lower Open-to-Close Return', 'Upper Open-to-Close Return'):
        if column in range_summary.columns:
            range_summary[column] = range_summary[column].map(
                lambda value: f'{value:.2%}' if pd.notna(value) else value
            )
    for column in ('Lower Projected Close', 'Upper Projected Close'):
        if column in range_summary.columns:
            range_summary[column] = range_summary[column].map(
                lambda value: f'${value:,.2f}' if pd.notna(value) else value
            )
    display(range_summary)

if not tail_summary.empty:
    for column in ('Open-to-Close Long VaR Floor', 'Open-to-Close Long CVaR Floor', 'Open-to-Close Short VaR Ceiling', 'Open-to-Close Short CVaR Ceiling'):
        if column in tail_summary.columns:
            tail_summary[column] = tail_summary[column].map(
                lambda value: f'${value:,.2f}' if pd.notna(value) else value
            )
    display(tail_summary)

trade_range_fig = lineChartPlotter.plot_trade_range_probability_cone(
    cone_context={
        'cone_contexts_by_window': trade_range_cone_contexts_by_window,
        'windows': trade_range_window_options,
        'default_window': trade_range_window,
    },
    ticker_label=ticker_str,
)
trade_range_combined_fig = _stack_trade_range_figures(
    trade_range_history_fig,
    trade_range_fig,
    title_text=f'{ticker_str} Historical Two-Sided Trade Range Profile and Current Cone ({trade_range_window}-Session Window)',
)
show_plotly_figure(trade_range_combined_fig)


In [ ]:
# Block 8: GARCH(1,1)-normal open-to-close trade range variant

# Refresh the shared plotter so notebook reruns pick up local code edits without a kernel restart.
from importlib import reload
import Quantapp.visualization.line_chart_plotter as _line_chart_plotter_module
reload(_line_chart_plotter_module)
lineChartPlotter = _line_chart_plotter_module.LineChartPlotter()

from scipy.stats import norm

try:
    from arch import arch_model
except ModuleNotFoundError as exc:
    if exc.name != 'arch':
        raise
    raise ImportError(
        "Block 8 requires the 'arch' package. Install it in the notebook kernel environment with `pip install arch`."
    ) from exc

trade_range_garch_window = trade_range_default_window if 'trade_range_default_window' in globals() else 200
trade_range_garch_window_options = trade_range_window_options if 'trade_range_window_options' in globals() else [trade_range_garch_window]
trade_range_garch_interval_levels = trade_range_interval_levels if 'trade_range_interval_levels' in globals() else [0.95, 0.99]
trade_range_garch_tail_levels = trade_range_tail_levels if 'trade_range_tail_levels' in globals() else [0.95, 0.99]

# Fit the model to completed open-to-close sessions only so the forecast stays aligned with Block 7.
garch_trade_frame = ticker[['Open', 'Close']].dropna().copy()
garch_session_returns = garch_trade_frame['Close'].div(garch_trade_frame['Open']).sub(1.0).dropna()
if len(garch_session_returns) < 3:
    raise ValueError(
        'Block 8 needs at least three sessions with valid open and close prices to fit a GARCH trade-range variant.'
    )

garch_completed_frame = garch_trade_frame.iloc[:-1].copy()
garch_completed_returns = garch_completed_frame['Close'].div(garch_completed_frame['Open']).sub(1.0).dropna()
if len(garch_completed_returns) < 50:
    raise ValueError(
        f'Block 8 needs at least 50 completed sessions to fit a stable GARCH trade-range variant. Only {len(garch_completed_returns)} are available.'
    )

# Historical diagnostics use a filtered full-sample GARCH fit across completed sessions.
garch_history_model = arch_model(
    garch_completed_returns.mul(100.0),
    mean='Constant',
    vol='GARCH',
    p=1,
    q=1,
    dist='normal',
    rescale=False,
)
garch_history_fit = garch_history_model.fit(disp='off')
garch_history_mean_return = float(garch_history_fit.params.get('mu', 0.0) / 100.0)
garch_history_sigma_series = pd.Series(
    garch_history_fit.conditional_volatility / 100.0,
    index=garch_completed_returns.index,
)
garch_history_open = garch_completed_frame['Open'].astype(float).reindex(garch_completed_returns.index)
garch_history_close = garch_completed_frame['Close'].astype(float).reindex(garch_completed_returns.index)
garch_current_history_forecast = garch_history_fit.forecast(horizon=1, reindex=False)
garch_current_history_mean_return = float(garch_current_history_forecast.mean.iloc[-1, 0] / 100.0)
garch_current_history_sigma_return = float(np.sqrt(garch_current_history_forecast.variance.iloc[-1, 0]) / 100.0)
if not np.isfinite(garch_current_history_sigma_return) or garch_current_history_sigma_return <= 0:
    raise ValueError(
        'Block 8 could not produce a positive current-session GARCH volatility forecast for the historical diagnostics.'
    )
garch_current_history_date = garch_trade_frame.index[-1]
garch_current_history_return = float(garch_session_returns.iloc[-1])
garch_history_metrics_by_confidence = {}
for confidence in sorted(set(trade_range_garch_interval_levels).union(trade_range_garch_tail_levels)):
    alpha = 1.0 - confidence
    interval_alpha = (1.0 - confidence) / 2.0
    interval_z = float(norm.ppf(interval_alpha))
    z_alpha = float(norm.ppf(alpha))
    pdf_alpha = float(norm.pdf(z_alpha))

    lower_interval_return = garch_history_mean_return + (garch_history_sigma_series * interval_z)
    upper_interval_return = garch_history_mean_return - (garch_history_sigma_series * interval_z)
    lower_var_return = garch_history_mean_return + (garch_history_sigma_series * z_alpha)
    upper_var_return = garch_history_mean_return - (garch_history_sigma_series * z_alpha)
    lower_expected_shortfall_return = garch_history_mean_return - (garch_history_sigma_series * (pdf_alpha / alpha))
    upper_expected_shortfall_return = garch_history_mean_return + (garch_history_sigma_series * (pdf_alpha / alpha))

    current_lower_var_return = garch_current_history_mean_return + (garch_current_history_sigma_return * z_alpha)
    current_upper_var_return = garch_current_history_mean_return - (garch_current_history_sigma_return * z_alpha)
    current_lower_expected_shortfall_return = (
        garch_current_history_mean_return - (garch_current_history_sigma_return * (pdf_alpha / alpha))
    )
    current_upper_expected_shortfall_return = (
        garch_current_history_mean_return + (garch_current_history_sigma_return * (pdf_alpha / alpha))
    )

    lower_var_return_with_current = pd.concat([
        lower_var_return,
        pd.Series([current_lower_var_return], index=[garch_current_history_date]),
    ]).sort_index()
    upper_var_return_with_current = pd.concat([
        upper_var_return,
        pd.Series([current_upper_var_return], index=[garch_current_history_date]),
    ]).sort_index()
    lower_expected_shortfall_return_with_current = pd.concat([
        lower_expected_shortfall_return,
        pd.Series([current_lower_expected_shortfall_return], index=[garch_current_history_date]),
    ]).sort_index()
    upper_expected_shortfall_return_with_current = pd.concat([
        upper_expected_shortfall_return,
        pd.Series([current_upper_expected_shortfall_return], index=[garch_current_history_date]),
    ]).sort_index()

    lower_breaches = garch_completed_returns.lt(lower_var_return).astype(float)
    upper_breaches = garch_completed_returns.gt(upper_var_return).astype(float)
    lower_breaches_with_current = pd.concat([
        lower_breaches,
        pd.Series([float(garch_current_history_return < current_lower_var_return)], index=[garch_current_history_date]),
    ]).sort_index()
    upper_breaches_with_current = pd.concat([
        upper_breaches,
        pd.Series([float(garch_current_history_return > current_upper_var_return)], index=[garch_current_history_date]),
    ]).sort_index()
    lower_rolling_breach_rate = lower_breaches_with_current.rolling(trade_range_garch_window).mean().dropna()
    upper_rolling_breach_rate = upper_breaches_with_current.rolling(trade_range_garch_window).mean().dropna()

    lower_expected_breach_rate = pd.Series(
        data=np.full(len(lower_rolling_breach_rate.index), alpha, dtype=float),
        index=lower_rolling_breach_rate.index,
    )
    upper_expected_breach_rate = pd.Series(
        data=np.full(len(upper_rolling_breach_rate.index), alpha, dtype=float),
        index=upper_rolling_breach_rate.index,
    )

    open_for_interval = garch_history_open.reindex(lower_interval_return.index)
    open_for_var = garch_history_open.reindex(lower_var_return.index)
    open_for_es = garch_history_open.reindex(lower_expected_shortfall_return.index)

    garch_history_metrics_by_confidence[confidence] = {
        'session_returns': garch_completed_returns,
        'session_open': garch_history_open,
        'session_close': garch_history_close,
        'lower_interval_return': lower_interval_return,
        'upper_interval_return': upper_interval_return,
        'lower_interval_price': open_for_interval.mul(1.0 + lower_interval_return),
        'upper_interval_price': open_for_interval.mul(1.0 + upper_interval_return),
        'lower_var_return': lower_var_return_with_current,
        'upper_var_return': upper_var_return_with_current,
        'lower_var_price': open_for_var.mul(1.0 + lower_var_return),
        'upper_var_price': open_for_var.mul(1.0 + upper_var_return),
        'lower_expected_shortfall_return': lower_expected_shortfall_return_with_current,
        'upper_expected_shortfall_return': upper_expected_shortfall_return_with_current,
        'lower_expected_shortfall_price': open_for_es.mul(1.0 + lower_expected_shortfall_return),
        'upper_expected_shortfall_price': open_for_es.mul(1.0 + upper_expected_shortfall_return),
        'lower_breaches': lower_breaches_with_current,
        'upper_breaches': upper_breaches_with_current,
        'lower_rolling_breach_rate': lower_rolling_breach_rate,
        'upper_rolling_breach_rate': upper_rolling_breach_rate,
        'lower_expected_breach_rate': lower_expected_breach_rate,
        'upper_expected_breach_rate': upper_expected_breach_rate,
    }

garch_trade_history_context = {
    'window': trade_range_garch_window,
    'interval_confidence_levels': trade_range_garch_interval_levels,
    'tail_confidence_levels': trade_range_garch_tail_levels,
    'metrics_by_confidence': garch_history_metrics_by_confidence,
    'session_returns': garch_session_returns,
    'session_open': garch_trade_frame['Open'].astype(float),
    'session_close': garch_trade_frame['Close'].astype(float),
}
garch_trade_history_fig = lineChartPlotter.plot_trade_range_history_profile(
    history_context=garch_trade_history_context,
    ticker_label=f'{ticker_str} GARCH(1,1) Normal',
)
from plotly.subplots import make_subplots
import copy as _copy


def _axis_index_from_ref(axis_ref, axis_letter):
    if axis_ref in (None, axis_letter):
        return 1
    if isinstance(axis_ref, str) and axis_ref.startswith(axis_letter):
        digits = ''.join(ch for ch in axis_ref[1:] if ch.isdigit())
        return int(digits) if digits else 1
    return 1


def _shift_axis_ref(axis_ref, row_offset):
    if not isinstance(axis_ref, str) or axis_ref == 'paper':
        return axis_ref
    suffix = ' domain' if axis_ref.endswith(' domain') else ''
    base = axis_ref[:-7] if suffix else axis_ref
    if not base or base[0] not in ('x', 'y'):
        return axis_ref
    axis_letter = base[0]
    digits = base[1:]
    axis_index = int(digits) if digits else 1
    shifted_index = axis_index + row_offset
    shifted_base = axis_letter if shifted_index == 1 else f'{axis_letter}{shifted_index}'
    return shifted_base + suffix


def _stack_trade_range_figures(history_fig, cone_fig, *, title_text):
    combined_fig = make_subplots(
        rows=7,
        cols=1,
        shared_xaxes=False,
        vertical_spacing=0.04,
        row_heights=[0.16, 0.13, 0.12, 0.12, 0.14, 0.14, 0.19],
        subplot_titles=(
            'Open-to-Close Returns with Two-Sided Tail Thresholds',
            'Rolling Open-to-Close Tail Forecasts: Long Floors and Short Ceilings',
            'Rolling Open-to-Close Lower Breach Rates vs Expected',
            'Rolling Open-to-Close Upper Breach Rates vs Expected',
            'Today Open-Anchored 95% Projected Close Range (Open-to-Close)',
            'Today Open-Anchored 99% Projected Close Range (Open-to-Close)',
            'Trailing Open-to-Close Return Distribution',
        ),
    )
    subplot_title_annotations = [_copy.deepcopy(annotation).to_plotly_json() for annotation in combined_fig.layout.annotations]

    def _to_json(item):
        payload = item.to_plotly_json() if hasattr(item, 'to_plotly_json') else dict(item)
        return _copy.deepcopy(payload)

    def _shift_shapes(shapes, row_offset):
        shifted_shapes = []
        for shape in shapes:
            shape_payload = _to_json(shape)
            shape_payload['xref'] = _shift_axis_ref(shape_payload.get('xref'), row_offset)
            shape_payload['yref'] = _shift_axis_ref(shape_payload.get('yref'), row_offset)
            shifted_shapes.append(shape_payload)
        return shifted_shapes

    def _shift_annotations(annotations, row_offset):
        shifted_annotations = []
        for annotation in annotations:
            annotation_payload = _to_json(annotation)
            annotation_payload['xref'] = _shift_axis_ref(annotation_payload.get('xref'), row_offset)
            annotation_payload['yref'] = _shift_axis_ref(annotation_payload.get('yref'), row_offset)
            shifted_annotations.append(annotation_payload)
        return shifted_annotations

    def _window_title(base_title, window_label):
        suffix = '-Session Window)'
        marker = base_title.rfind('(')
        suffix_index = base_title.rfind(suffix)
        if marker != -1 and suffix_index != -1 and marker < suffix_index:
            return f"{base_title[:marker + 1]}{window_label}{suffix}"
        return base_title

    for trace in history_fig.data:
        target_row = _axis_index_from_ref(getattr(trace, 'yaxis', None), 'y')
        stacked_trace = _copy.deepcopy(trace)
        stacked_trace.xaxis = None
        stacked_trace.yaxis = None
        combined_fig.add_trace(stacked_trace, row=target_row, col=1)

    for trace in cone_fig.data:
        target_row = 4 + _axis_index_from_ref(getattr(trace, 'yaxis', None), 'y')
        stacked_trace = _copy.deepcopy(trace)
        stacked_trace.xaxis = None
        stacked_trace.yaxis = None
        combined_fig.add_trace(stacked_trace, row=target_row, col=1)

    history_static_shapes = _shift_shapes(history_fig.layout.shapes, 0)
    history_extra_annotations = _shift_annotations(history_fig.layout.annotations[4:], 0)
    cone_default_shapes = _shift_shapes(cone_fig.layout.shapes, 4)
    cone_default_annotations = _shift_annotations(cone_fig.layout.annotations[3:], 4)

    combined_fig.update_yaxes(title_text=history_fig.layout.yaxis.title.text, tickformat=history_fig.layout.yaxis.tickformat, row=1, col=1)
    combined_fig.update_yaxes(title_text=history_fig.layout.yaxis2.title.text, tickformat=history_fig.layout.yaxis2.tickformat, row=2, col=1)
    combined_fig.update_yaxes(title_text=history_fig.layout.yaxis3.title.text, tickformat=history_fig.layout.yaxis3.tickformat, row=3, col=1)
    combined_fig.update_yaxes(title_text=history_fig.layout.yaxis4.title.text, tickformat=history_fig.layout.yaxis4.tickformat, row=4, col=1)
    if history_fig.layout.xaxis4.range:
        combined_fig.update_xaxes(range=list(history_fig.layout.xaxis4.range), row=4, col=1)
    combined_fig.update_xaxes(title_text=history_fig.layout.xaxis4.title.text, row=4, col=1)

    combined_fig.update_xaxes(range=list(cone_fig.layout.xaxis.range), tickmode=cone_fig.layout.xaxis.tickmode, tickvals=list(cone_fig.layout.xaxis.tickvals), ticktext=list(cone_fig.layout.xaxis.ticktext), title_text=cone_fig.layout.xaxis.title.text, row=5, col=1)
    combined_fig.update_yaxes(title_text=cone_fig.layout.yaxis.title.text, range=list(cone_fig.layout.yaxis.range), tickprefix=cone_fig.layout.yaxis.tickprefix, row=5, col=1)
    combined_fig.update_xaxes(range=list(cone_fig.layout.xaxis2.range), tickmode=cone_fig.layout.xaxis2.tickmode, tickvals=list(cone_fig.layout.xaxis2.tickvals), ticktext=list(cone_fig.layout.xaxis2.ticktext), title_text=cone_fig.layout.xaxis2.title.text, row=6, col=1)
    combined_fig.update_yaxes(title_text=cone_fig.layout.yaxis2.title.text, range=list(cone_fig.layout.yaxis2.range), tickprefix=cone_fig.layout.yaxis2.tickprefix, row=6, col=1)
    combined_fig.update_xaxes(title_text=cone_fig.layout.xaxis3.title.text, ticksuffix=cone_fig.layout.xaxis3.ticksuffix, row=7, col=1)
    combined_fig.update_yaxes(title_text=cone_fig.layout.yaxis3.title.text, row=7, col=1)

    updatemenus = []
    history_menu = history_fig.layout.updatemenus[0] if getattr(history_fig.layout, 'updatemenus', None) else None
    cone_menu = cone_fig.layout.updatemenus[0] if getattr(cone_fig.layout, 'updatemenus', None) else None
    active_index = 0
    initial_title_text = title_text

    if history_menu is not None and cone_menu is not None:
        history_buttons = list(history_menu.buttons)
        cone_buttons = list(cone_menu.buttons)
        if len(history_buttons) == len(cone_buttons) and history_buttons:
            active_index = int(history_menu.active) if history_menu.active is not None else 0
            active_index = max(0, min(active_index, len(history_buttons) - 1))
            combined_buttons = []

            for history_button, cone_button in zip(history_buttons, cone_buttons):
                history_label = str(history_button.label)
                cone_label = str(cone_button.label)
                if history_label != cone_label:
                    raise ValueError('History and cone dropdown labels do not align for stacked trade-range figure.')

                history_args = history_button.args if history_button.args is not None else []
                cone_args = cone_button.args if cone_button.args is not None else []
                history_visibility = list(history_args[0].get('visible', [])) if history_args else []
                cone_visibility = list(cone_args[0].get('visible', [])) if cone_args else []
                if not history_visibility:
                    history_visibility = [trace.visible if trace.visible is not None else True for trace in history_fig.data]
                if not cone_visibility:
                    cone_visibility = [trace.visible if trace.visible is not None else True for trace in cone_fig.data]

                cone_layout_updates = cone_args[1] if len(cone_args) > 1 else {}
                cone_shapes = _shift_shapes(cone_layout_updates.get('shapes', cone_fig.layout.shapes), 4)
                cone_annotations = _shift_annotations(cone_layout_updates.get('annotations', cone_fig.layout.annotations)[3:], 4)
                combined_buttons.append(
                    dict(
                        label=history_label,
                        method='update',
                        args=[
                            {'visible': history_visibility + cone_visibility},
                            {
                                'title': lineChartPlotter._header_title(_window_title(title_text, history_label)),
                                'shapes': history_static_shapes + cone_shapes,
                                'annotations': subplot_title_annotations + history_extra_annotations + cone_annotations,
                            },
                        ],
                    )
                )

            initial_title_text = _window_title(title_text, str(history_buttons[active_index].label))
            updatemenus = [
                lineChartPlotter._dropdown_menu(
                    buttons=combined_buttons,
                    x=0.0,
                    active=active_index,
                )
            ]

    combined_fig.update_layout(
        title=lineChartPlotter._header_title(initial_title_text),
        height=2550,
        margin=lineChartPlotter._header_margin(top=180),
        template='plotly_dark',
        hovermode='closest',
        legend=dict(orientation='h', yanchor='bottom', y=1.01, xanchor='left', x=0),
        bargap=0.08,
        updatemenus=updatemenus,
        shapes=history_static_shapes + cone_default_shapes,
        annotations=subplot_title_annotations + history_extra_annotations + cone_default_annotations,
    )
    return combined_fig


# Current-session forecast uses the selected trailing fit window so the cone can switch windows like Block 7.
def build_garch_trade_range_cone_components(window):
    garch_forecast_returns = garch_completed_returns.tail(window).dropna()
    garch_effective_window = len(garch_forecast_returns)
    if garch_effective_window < 21:
        raise ValueError(
            f'Block 8 needs at least 21 completed sessions inside the trailing fit window. Only {garch_effective_window} are available for the {window}-session view.'
        )

    garch_trade_model = arch_model(
        garch_forecast_returns.mul(100.0),
        mean='Constant',
        vol='GARCH',
        p=1,
        q=1,
        dist='normal',
        rescale=False,
    )
    garch_trade_fit = garch_trade_model.fit(disp='off')
    garch_trade_forecast = garch_trade_fit.forecast(horizon=1, reindex=False)

    garch_mean_return = float(garch_trade_forecast.mean.iloc[-1, 0] / 100.0)
    garch_sigma_return = float(np.sqrt(garch_trade_forecast.variance.iloc[-1, 0]) / 100.0)
    if not np.isfinite(garch_sigma_return) or garch_sigma_return <= 0:
        raise ValueError(
            f'Block 8 could not produce a positive one-step-ahead GARCH volatility forecast for the {window}-session view.'
        )

    garch_anchor_price = float(garch_trade_frame['Open'].iloc[-1])
    garch_latest_price = float(garch_trade_frame['Close'].iloc[-1])
    garch_session_date = garch_trade_frame.index[-1]
    garch_median_return = garch_mean_return
    garch_median_price = garch_anchor_price * (1.0 + garch_median_return)

    garch_model_summary = pd.DataFrame([
        {
            'Model': 'GARCH(1,1) Normal',
            'Fit Window': garch_effective_window,
            'Forecast Mean Open-to-Close Return': garch_mean_return,
            'Forecast Sigma Open-to-Close Return': garch_sigma_return,
            'Omega': float(garch_trade_fit.params.get('omega', np.nan)),
            'Alpha(1)': float(garch_trade_fit.params.get('alpha[1]', np.nan)),
            'Beta(1)': float(garch_trade_fit.params.get('beta[1]', np.nan)),
            'Alpha + Beta': float(garch_trade_fit.params.get('alpha[1]', np.nan) + garch_trade_fit.params.get('beta[1]', np.nan)),
        }
    ])

    garch_interval_map = {}
    garch_range_rows = []
    for confidence in trade_range_garch_interval_levels:
        tail_probability = (1.0 - confidence) / 2.0
        lower_return = float(norm.ppf(tail_probability, loc=garch_mean_return, scale=garch_sigma_return))
        upper_return = float(norm.ppf(1.0 - tail_probability, loc=garch_mean_return, scale=garch_sigma_return))
        lower_price = garch_anchor_price * (1.0 + lower_return)
        upper_price = garch_anchor_price * (1.0 + upper_return)
        garch_interval_map[confidence] = {
            'lower_return': lower_return,
            'upper_return': upper_return,
            'lower_price': lower_price,
            'upper_price': upper_price,
        }
        garch_range_rows.append(
            {
                'Confidence': f'{confidence:.0%}',
                'Lower Return': lower_return,
                'Upper Return': upper_return,
                'Lower Close': lower_price,
                'Upper Close': upper_price,
            }
        )

    garch_long_tail_map = {}
    garch_short_tail_map = {}
    garch_tail_rows = []
    for confidence in trade_range_garch_tail_levels:
        alpha = 1.0 - confidence
        z_alpha = float(norm.ppf(alpha))
        pdf_alpha = float(norm.pdf(z_alpha))

        long_var_return = garch_mean_return + (garch_sigma_return * z_alpha)
        long_cvar_return = garch_mean_return - (garch_sigma_return * (pdf_alpha / alpha))
        short_var_return = garch_mean_return - (garch_sigma_return * z_alpha)
        short_cvar_return = garch_mean_return + (garch_sigma_return * (pdf_alpha / alpha))

        garch_long_tail_map[confidence] = {
            'var_return': long_var_return,
            'var_price': garch_anchor_price * (1.0 + long_var_return),
            'expected_shortfall_return': long_cvar_return,
            'expected_shortfall_price': garch_anchor_price * (1.0 + long_cvar_return),
        }
        garch_short_tail_map[confidence] = {
            'var_return': short_var_return,
            'var_price': garch_anchor_price * (1.0 + short_var_return),
            'expected_shortfall_return': short_cvar_return,
            'expected_shortfall_price': garch_anchor_price * (1.0 + short_cvar_return),
        }
        garch_tail_rows.append(
            {
                'Confidence': f'{confidence:.0%}',
                'Long VaR Floor': garch_long_tail_map[confidence]['var_price'],
                'Long CVaR Floor': garch_long_tail_map[confidence]['expected_shortfall_price'],
                'Short VaR Ceiling': garch_short_tail_map[confidence]['var_price'],
                'Short CVaR Ceiling': garch_short_tail_map[confidence]['expected_shortfall_price'],
            }
        )

    garch_range_summary = pd.DataFrame(garch_range_rows).rename(columns={
        'Confidence': 'GARCH Open-to-Close Confidence',
        'Lower Return': 'Lower GARCH Open-to-Close Return',
        'Upper Return': 'Upper GARCH Open-to-Close Return',
        'Lower Close': 'Lower GARCH Projected Close',
        'Upper Close': 'Upper GARCH Projected Close',
    })
    garch_tail_summary = pd.DataFrame(garch_tail_rows).rename(columns={
        'Confidence': 'GARCH Tail Confidence',
        'Long VaR Floor': 'GARCH Open-to-Close Long VaR Floor',
        'Long CVaR Floor': 'GARCH Open-to-Close Long CVaR Floor',
        'Short VaR Ceiling': 'GARCH Open-to-Close Short VaR Ceiling',
        'Short CVaR Ceiling': 'GARCH Open-to-Close Short CVaR Ceiling',
    })

    garch_trade_range_context = {
        'session_date': garch_session_date,
        'window': window,
        'effective_window': garch_effective_window,
        'anchor_price': garch_anchor_price,
        'latest_price': garch_latest_price,
        'sample_returns': garch_forecast_returns,
        'interval_confidence_levels': trade_range_garch_interval_levels,
        'tail_confidence_levels': trade_range_garch_tail_levels,
        'intervals': garch_interval_map,
        'long_tail_levels': garch_long_tail_map,
        'short_tail_levels': garch_short_tail_map,
        'median_return': garch_median_return,
        'median_price': garch_median_price,
    }

    return {
        'context': garch_trade_range_context,
        'model_summary': garch_model_summary,
        'range_summary': garch_range_summary,
        'tail_summary': garch_tail_summary,
    }

garch_cone_components_by_window = {
    window: build_garch_trade_range_cone_components(window)
    for window in trade_range_garch_window_options
}
garch_default_components = garch_cone_components_by_window[trade_range_garch_window]

formatted_garch_model_summary = garch_default_components['model_summary'].copy()
for column in ('Forecast Mean Open-to-Close Return', 'Forecast Sigma Open-to-Close Return'):
    formatted_garch_model_summary[column] = formatted_garch_model_summary[column].map(
        lambda value: f'{value:.2%}' if pd.notna(value) else value
    )
for column in ('Omega', 'Alpha(1)', 'Beta(1)', 'Alpha + Beta'):
    formatted_garch_model_summary[column] = formatted_garch_model_summary[column].map(
        lambda value: f'{value:.4f}' if pd.notna(value) else value
    )
display(formatted_garch_model_summary)

garch_range_summary = garch_default_components['range_summary'].copy()
if not garch_range_summary.empty:
    for column in ('Lower GARCH Open-to-Close Return', 'Upper GARCH Open-to-Close Return'):
        if column in garch_range_summary.columns:
            garch_range_summary[column] = garch_range_summary[column].map(
                lambda value: f'{value:.2%}' if pd.notna(value) else value
            )
    for column in ('Lower GARCH Projected Close', 'Upper GARCH Projected Close'):
        if column in garch_range_summary.columns:
            garch_range_summary[column] = garch_range_summary[column].map(
                lambda value: f'${value:,.2f}' if pd.notna(value) else value
            )
    display(garch_range_summary)

garch_tail_summary = garch_default_components['tail_summary'].copy()
if not garch_tail_summary.empty:
    for column in (
        'GARCH Open-to-Close Long VaR Floor',
        'GARCH Open-to-Close Long CVaR Floor',
        'GARCH Open-to-Close Short VaR Ceiling',
        'GARCH Open-to-Close Short CVaR Ceiling',
    ):
        if column in garch_tail_summary.columns:
            garch_tail_summary[column] = garch_tail_summary[column].map(
                lambda value: f'${value:,.2f}' if pd.notna(value) else value
            )
    display(garch_tail_summary)

garch_trade_range_fig = lineChartPlotter.plot_trade_range_probability_cone(
    cone_context={
        'cone_contexts_by_window': {
            window: components['context']
            for window, components in garch_cone_components_by_window.items()
        },
        'windows': trade_range_garch_window_options,
        'default_window': trade_range_garch_window,
    },
    ticker_label=f'{ticker_str} GARCH(1,1)',
)
garch_trade_combined_fig = _stack_trade_range_figures(
    garch_trade_history_fig,
    garch_trade_range_fig,
    title_text=f'{ticker_str} GARCH(1,1) Historical Trade Range Profile and Current Cone ({trade_range_garch_window}-Session Window)',
)
show_plotly_figure(garch_trade_combined_fig)


In [ ]:
# Block 8B: preview the combined trade-range selector with a method dropdown

import ipywidgets as widgets
from IPython.display import clear_output, display

trade_range_preview_views = {}

if all(name in globals() for name in ('range_summary', 'tail_summary', 'trade_range_combined_fig')):
    trade_range_preview_views['Empirical'] = {
        'caption': 'Empirical open-to-close trade range using completed-session return history.',
        'model_summary': None,
        'range_summary': range_summary.copy(),
        'tail_summary': tail_summary.copy(),
        'figure': trade_range_combined_fig,
    }

if all(name in globals() for name in ('formatted_garch_model_summary', 'garch_range_summary', 'garch_tail_summary', 'garch_trade_combined_fig')):
    trade_range_preview_views['GARCH(1,1) Normal'] = {
        'caption': 'GARCH(1,1)-normal trade range fit on completed open-to-close sessions.',
        'model_summary': formatted_garch_model_summary.copy(),
        'range_summary': garch_range_summary.copy(),
        'tail_summary': garch_tail_summary.copy(),
        'figure': garch_trade_combined_fig,
    }

if not trade_range_preview_views:
    raise ValueError('Run Block 7 first. Run Block 8 as well if you want the GARCH option in this preview.')

trade_range_preview_dropdown = widgets.Dropdown(
    options=list(trade_range_preview_views.keys()),
    value='Empirical' if 'Empirical' in trade_range_preview_views else next(iter(trade_range_preview_views)),
    description='Method:',
    layout=widgets.Layout(width='320px'),
)
trade_range_preview_output = widgets.Output()
trade_range_preview_show_figure = show_plotly_figure if 'show_plotly_figure' in globals() else (lambda fig: fig.show())


def render_trade_range_preview(*_):
    selected_view = trade_range_preview_views[trade_range_preview_dropdown.value]

    with trade_range_preview_output:
        clear_output(wait=True)
        display(widgets.HTML(
            '<div style="margin: 0 0 10px 0;"><b>Preview:</b> ' + selected_view['caption'] + '</div>'
        ))

        if selected_view['model_summary'] is not None:
            display(selected_view['model_summary'])

        display(selected_view['range_summary'])
        display(selected_view['tail_summary'])
        trade_range_preview_show_figure(selected_view['figure'])


trade_range_preview_dropdown.observe(render_trade_range_preview, names='value')
display(widgets.VBox([trade_range_preview_dropdown, trade_range_preview_output]))
render_trade_range_preview()


In [ ]:
# Block 9: backtest fixed payout structures against the 95% long confidence-interval floor

from plotly.subplots import make_subplots

strategy_payout_options = [(float(risk_dollars), float(100 - risk_dollars)) for risk_dollars in range(10, 100, 10)]
strategy_default_payout = (70.0, 30.0)
strategy_confidence = 0.95
strategy_rolling_pnl_window = 21


def format_strategy_payout_label(risk_dollars, reward_dollars):
    return f'Risk ${risk_dollars:,.0f} / Reward ${reward_dollars:,.0f}'


def format_strategy_payout_short_label(risk_dollars, reward_dollars):
    return f'{risk_dollars:,.0f}/{reward_dollars:,.0f}'


def build_fixed_payout_trade_profile(strategy_label, metrics_by_confidence, *, confidence, risk_dollars, reward_dollars, window_label, rolling_window):
    metric_set = metrics_by_confidence.get(confidence)
    if metric_set is None:
        raise KeyError(f'{strategy_label} does not contain a {confidence:.0%} long interval-floor series.')

    long_interval_floor = pd.Series(metric_set['lower_interval_price']).dropna()
    session_open = pd.Series(metric_set['session_open']).reindex(long_interval_floor.index)
    session_close = pd.Series(metric_set['session_close']).reindex(long_interval_floor.index)
    session_returns = pd.Series(metric_set['session_returns']).reindex(long_interval_floor.index)

    trade_frame = pd.DataFrame({
        'Open': session_open,
        'Close': session_close,
        'Open-to-Close Return': session_returns,
        'Long 95% Interval Floor': long_interval_floor,
    }).dropna()
    if trade_frame.empty:
        raise ValueError(f'{strategy_label} does not contain any fully aligned historical trades to evaluate.')

    breakeven_win_rate = risk_dollars / (risk_dollars + reward_dollars)
    payout_label = format_strategy_payout_label(risk_dollars, reward_dollars)

    trade_frame['Win'] = trade_frame['Close'].ge(trade_frame['Long 95% Interval Floor'])
    trade_frame['Outcome'] = np.where(trade_frame['Win'], 'Win', 'Loss')
    trade_frame['Trade PnL'] = np.where(trade_frame['Win'], reward_dollars, -risk_dollars)
    trade_frame['Rolling PnL'] = trade_frame['Trade PnL'].rolling(rolling_window, min_periods=1).sum()
    trade_frame['Rolling Win Rate'] = trade_frame['Win'].rolling(rolling_window, min_periods=1).mean()
    trade_frame['Cumulative PnL'] = trade_frame['Trade PnL'].cumsum()
    trade_frame['Running Peak PnL'] = trade_frame['Cumulative PnL'].cummax()
    trade_frame['Drawdown'] = trade_frame['Cumulative PnL'] - trade_frame['Running Peak PnL']
    trade_frame['Payout Structure'] = payout_label

    gross_profit = float(trade_frame.loc[trade_frame['Trade PnL'] > 0, 'Trade PnL'].sum())
    gross_loss = float(-trade_frame.loc[trade_frame['Trade PnL'] < 0, 'Trade PnL'].sum())
    profit_factor = np.nan if gross_loss == 0 else gross_profit / gross_loss

    summary = {
        'Payout Structure': payout_label,
        'Strategy': strategy_label,
        'Window': window_label,
        'Confidence': f'{confidence:.0%}',
        'Trades': int(len(trade_frame)),
        'Wins': int(trade_frame['Win'].sum()),
        'Losses': int((~trade_frame['Win']).sum()),
        'Win Rate': float(trade_frame['Win'].mean()),
        'Breakeven Win Rate': float(breakeven_win_rate),
        'Average Trade PnL': float(trade_frame['Trade PnL'].mean()),
        'Latest Rolling PnL': float(trade_frame['Rolling PnL'].iloc[-1]),
        'Total PnL': float(trade_frame['Trade PnL'].sum()),
        'Profit Factor': profit_factor,
        'Worst Drawdown': float(trade_frame['Drawdown'].min()),
    }

    return {
        'label': strategy_label,
        'summary': summary,
        'trade_frame': trade_frame,
    }


strategy_include_garch = 'garch_trade_history_context' in globals()
if not strategy_include_garch:
    print('Run Block 8 first if you want the GARCH strategy included in this fixed-payout evaluation.')


def build_strategy_profiles_for_payout(risk_dollars, reward_dollars):
    strategy_profiles = []
    for window in trade_range_history_context['windows']:
        strategy_profiles.append(
            build_fixed_payout_trade_profile(
                strategy_label=f'Current Method ({window}-Session)',
                metrics_by_confidence=trade_range_history_context['metrics_by_window'][window],
                confidence=strategy_confidence,
                risk_dollars=risk_dollars,
                reward_dollars=reward_dollars,
                window_label=window,
                rolling_window=strategy_rolling_pnl_window,
            )
        )

    if strategy_include_garch:
        garch_window_label = garch_trade_history_context.get('window', trade_range_default_window)
        strategy_profiles.append(
            build_fixed_payout_trade_profile(
                strategy_label=f'GARCH(1,1) Normal ({garch_window_label}-Session)',
                metrics_by_confidence=garch_trade_history_context['metrics_by_confidence'],
                confidence=strategy_confidence,
                risk_dollars=risk_dollars,
                reward_dollars=reward_dollars,
                window_label=garch_window_label,
                rolling_window=strategy_rolling_pnl_window,
            )
        )

    return strategy_profiles


strategy_default_payout_label = format_strategy_payout_label(*strategy_default_payout)
strategy_profiles_by_payout = {}
strategy_summary_by_payout = {}
strategy_backtest_trade_logs_by_payout = {}
strategy_backtest_equity_curve_by_payout = {}
strategy_backtest_rolling_pnl_by_payout = {}

for risk_dollars, reward_dollars in strategy_payout_options:
    payout_label = format_strategy_payout_label(risk_dollars, reward_dollars)
    strategy_profiles = build_strategy_profiles_for_payout(risk_dollars, reward_dollars)
    strategy_profiles_by_payout[payout_label] = strategy_profiles
    strategy_summary_by_payout[payout_label] = pd.DataFrame([profile['summary'] for profile in strategy_profiles])
    strategy_backtest_trade_logs_by_payout[payout_label] = {
        profile['label']: profile['trade_frame'].copy()
        for profile in strategy_profiles
    }
    strategy_backtest_equity_curve_by_payout[payout_label] = pd.concat(
        {profile['label']: profile['trade_frame']['Cumulative PnL'] for profile in strategy_profiles},
        axis=1,
    ).sort_index()
    strategy_backtest_rolling_pnl_by_payout[payout_label] = pd.concat(
        {profile['label']: profile['trade_frame']['Rolling PnL'] for profile in strategy_profiles},
        axis=1,
    ).sort_index()

strategy_summary_lookup = pd.concat(strategy_summary_by_payout.values(), ignore_index=True)
formatted_strategy_summary_lookup = strategy_summary_lookup.copy()
for column in ('Win Rate', 'Breakeven Win Rate'):
    if column in formatted_strategy_summary_lookup.columns:
        formatted_strategy_summary_lookup[column] = formatted_strategy_summary_lookup[column].map(
            lambda value: f'{value:.2%}' if pd.notna(value) else value
        )
for column in ('Average Trade PnL', 'Latest Rolling PnL', 'Total PnL', 'Worst Drawdown'):
    if column in formatted_strategy_summary_lookup.columns:
        formatted_strategy_summary_lookup[column] = formatted_strategy_summary_lookup[column].map(
            lambda value: f'${value:,.2f}' if pd.notna(value) else value
        )
if 'Profit Factor' in formatted_strategy_summary_lookup.columns:
    formatted_strategy_summary_lookup['Profit Factor'] = formatted_strategy_summary_lookup['Profit Factor'].map(
        lambda value: 'N/A' if pd.isna(value) else f'{value:.2f}'
    )

display(formatted_strategy_summary_lookup)

strategy_backtest_trade_logs = strategy_backtest_trade_logs_by_payout[strategy_default_payout_label]
strategy_backtest_equity_curve = strategy_backtest_equity_curve_by_payout[strategy_default_payout_label]
strategy_backtest_rolling_pnl = strategy_backtest_rolling_pnl_by_payout[strategy_default_payout_label]
strategy_trade_log_preview = pd.concat(
    [
        profile['trade_frame'].assign(Strategy=profile['label'])
        for profile in strategy_profiles_by_payout[strategy_default_payout_label]
    ],
    axis=0,
).reset_index(names='Date')
strategy_trade_log_preview = strategy_trade_log_preview[['Date', 'Payout Structure', 'Strategy', 'Close', 'Long 95% Interval Floor', 'Outcome', 'Trade PnL', 'Rolling PnL', 'Rolling Win Rate', 'Cumulative PnL', 'Drawdown']]
display(strategy_trade_log_preview.tail(15))

strategy_backtest_fig = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.08,
    row_heights=[0.48, 0.52],
    subplot_titles=(
        f'{strategy_rolling_pnl_window}-Trade Rolling PnL',
        'Cumulative PnL',
    ),
)

strategy_trace_indices_by_payout = {}
for risk_dollars, reward_dollars in strategy_payout_options:
    payout_label = format_strategy_payout_label(risk_dollars, reward_dollars)
    payout_short_label = format_strategy_payout_short_label(risk_dollars, reward_dollars)
    strategy_profiles = strategy_profiles_by_payout[payout_label]
    strategy_trace_indices_by_payout[payout_label] = []
    is_default_payout = payout_label == strategy_default_payout_label

    for profile in strategy_profiles:
        trade_frame = profile['trade_frame']

        rolling_trace_index = len(strategy_backtest_fig.data)
        strategy_backtest_fig.add_trace(
            go.Scatter(
                x=trade_frame.index,
                y=trade_frame['Rolling PnL'],
                mode='lines',
                name=profile['label'],
                legendgroup=profile['label'],
                showlegend=True,
                visible=is_default_payout,
                customdata=np.column_stack([
                    trade_frame['Trade PnL'],
                    trade_frame['Rolling Win Rate'],
                    trade_frame['Cumulative PnL'],
                ]),
                hovertemplate=(
                    f'Payout: {payout_short_label}'
                    '<br>Date: %{x|%Y-%m-%d}'
                    '<br>Rolling PnL: $%{y:,.2f}'
                    '<br>Trade PnL: $%{customdata[0]:,.2f}'
                    '<br>Rolling Win Rate: %{customdata[1]:.2%}'
                    '<br>Cumulative PnL: $%{customdata[2]:,.2f}'
                    '<extra></extra>'
                ),
            ),
            row=1,
            col=1,
        )
        strategy_trace_indices_by_payout[payout_label].append(rolling_trace_index)

        cumulative_trace_index = len(strategy_backtest_fig.data)
        strategy_backtest_fig.add_trace(
            go.Scatter(
                x=trade_frame.index,
                y=trade_frame['Cumulative PnL'],
                mode='lines',
                name=profile['label'],
                legendgroup=profile['label'],
                showlegend=False,
                visible=is_default_payout,
                customdata=np.column_stack([
                    trade_frame['Trade PnL'],
                    trade_frame['Rolling PnL'],
                    trade_frame['Drawdown'],
                ]),
                hovertemplate=(
                    f'Payout: {payout_short_label}'
                    '<br>Date: %{x|%Y-%m-%d}'
                    '<br>Cumulative PnL: $%{y:,.2f}'
                    '<br>Trade PnL: $%{customdata[0]:,.2f}'
                    '<br>Rolling PnL: $%{customdata[1]:,.2f}'
                    '<br>Drawdown: $%{customdata[2]:,.2f}'
                    '<extra></extra>'
                ),
            ),
            row=2,
            col=1,
        )
        strategy_trace_indices_by_payout[payout_label].append(cumulative_trace_index)

strategy_backtest_fig.add_hline(
    y=0,
    line_dash='dash',
    line_color='rgba(248, 250, 252, 0.45)',
    line_width=1,
    row=1,
    col=1,
)
strategy_backtest_fig.add_hline(
    y=0,
    line_dash='dash',
    line_color='rgba(248, 250, 252, 0.45)',
    line_width=1,
    row=2,
    col=1,
)

strategy_dropdown_buttons = []
strategy_total_trace_count = len(strategy_backtest_fig.data)
for risk_dollars, reward_dollars in strategy_payout_options:
    payout_label = format_strategy_payout_label(risk_dollars, reward_dollars)
    visible_mask = [False] * strategy_total_trace_count
    for trace_index in strategy_trace_indices_by_payout[payout_label]:
        visible_mask[trace_index] = True

    strategy_dropdown_buttons.append(
        dict(
            label=format_strategy_payout_short_label(risk_dollars, reward_dollars),
            method='update',
            args=[
                {'visible': visible_mask},
                {
                    'title': lineChartPlotter._header_title(
                        f'{ticker_str} Fixed Payout Strategy vs 95% Long Interval Floor ({payout_label})'
                    )
                },
            ],
        )
    )

strategy_backtest_fig.update_layout(
    title=lineChartPlotter._header_title(
        f'{ticker_str} Fixed Payout Strategy vs 95% Long Interval Floor ({strategy_default_payout_label})'
    ),
    height=950,
    margin=lineChartPlotter._header_margin(),
    template='plotly_dark',
    hovermode='x unified',
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='left', x=0),
    updatemenus=[
        dict(
            type='dropdown',
            direction='down',
            buttons=strategy_dropdown_buttons,
            showactive=True,
            active=strategy_payout_options.index(strategy_default_payout),
            x=1.0,
            xanchor='right',
            y=1.18,
            yanchor='top',
            bgcolor='rgba(15, 23, 42, 0.95)',
            bordercolor='rgba(148, 163, 184, 0.45)',
            font=dict(size=11),
        )
    ],
    annotations=list(strategy_backtest_fig.layout.annotations) + [
        dict(
            x=1.0,
            y=1.205,
            xref='paper',
            yref='paper',
            xanchor='right',
            yanchor='bottom',
            showarrow=False,
            text='Risk / Reward',
            font=dict(size=11, color='rgba(226, 232, 240, 0.92)'),
        )
    ],
)
strategy_backtest_fig.update_xaxes(title_text='Date', row=2, col=1)
strategy_backtest_fig.update_yaxes(title_text=f'{strategy_rolling_pnl_window}-Trade Rolling PnL', tickprefix='$', row=1, col=1)
strategy_backtest_fig.update_yaxes(title_text='Cumulative PnL', tickprefix='$', row=2, col=1)
show_plotly_figure(strategy_backtest_fig)


In [ ]:
# Block 10: compare annualized GARCH-family volatility models to annualized rolling volatility estimators

import sys

def _purge_stale_modules(prefixes):
    for prefix in prefixes:
        matching_modules = [
            name
            for name in list(sys.modules)
            if name == prefix or name.startswith(f"{prefix}.")
        ]
        for module_name in matching_modules:
            sys.modules.pop(module_name, None)

try:
    from arch import arch_model
except ModuleNotFoundError as exc:
    if exc.name != "arch":
        raise
    raise ImportError(
        "Block 10 requires the 'arch' package. Install it in the notebook kernel environment with `pip install arch`."
    ) from exc
except Exception:
    _purge_stale_modules(("arch", "matplotlib"))
    try:
        from arch import arch_model
    except ModuleNotFoundError as exc:
        if exc.name != "arch":
            raise
        raise ImportError(
            "Block 10 requires the 'arch' package. Install it in the notebook kernel environment with `pip install arch`."
        ) from exc
    except Exception as exc:
        raise RuntimeError(
            "Block 10 could not import `arch` because the notebook kernel is holding a stale matplotlib state. Restart the kernel and rerun Block 2 if this persists."
        ) from exc

close_series = ticker["Close"]
ohlc_frame = ticker[["Open", "High", "Low", "Close"]].copy()
returns = close_series.pct_change()
garch_input = returns.dropna() * 100

log_hl = np.log(ohlc_frame["High"] / ohlc_frame["Low"])
log_ho = np.log(ohlc_frame["High"] / ohlc_frame["Open"])
log_lo = np.log(ohlc_frame["Low"] / ohlc_frame["Open"])
log_co = np.log(ohlc_frame["Close"] / ohlc_frame["Open"])
log_oc = np.log(ohlc_frame["Open"] / ohlc_frame["Close"].shift(1))
log_hc = np.log(ohlc_frame["High"] / ohlc_frame["Close"])
log_lc = np.log(ohlc_frame["Low"] / ohlc_frame["Close"])

garman_klass_variance = 0.5 * (log_hl ** 2) - ((2 * np.log(2)) - 1) * (log_co ** 2)
parkinson_variance = (log_hl ** 2) / (4 * np.log(2))
rs_variance = (log_hc * log_ho) + (log_lc * log_lo)

volatility_model_specs = [
    ("GARCH(1,1)", dict(vol="GARCH", p=1, q=1, o=0), "#111111", "solid"),
    ("EGARCH(1,1)", dict(vol="EGARCH", p=1, o=1, q=1), "#d62728", "dash"),
    ("GJR-GARCH(1,1)", dict(vol="GARCH", p=1, o=1, q=1), "#2ca02c", "dot"),
]

rolling_realized_vol_specs = [
    ("Close-to-Close", "close-to-close", "#1f77b4", "solid"),
    ("Parkinson", "parkinson", "#9467bd", "dash"),
    ("Yang-Zhang", "yang-zhang", "#ff7f0e", "dot"),
    ("Garman-Klass", "garman-klass", "#8c564b", "dashdot"),
    ("Rogers-Satchell", "rogers-satchell", "#17becf", "longdash"),
]

ewma_realized_vol_specs = [
    ("EWMA Close-to-Close", "close-to-close", "#1f77b4", "longdashdot"),
    ("EWMA Parkinson", "parkinson", "#9467bd", "longdashdot"),
    ("EWMA Yang-Zhang", "yang-zhang", "#ff7f0e", "longdashdot"),
    ("EWMA Garman-Klass", "garman-klass", "#8c564b", "longdashdot"),
    ("EWMA Rogers-Satchell", "rogers-satchell", "#17becf", "longdashdot"),
]

annualized_model_vols = {}
for model_name, model_kwargs, _, _ in volatility_model_specs:
    model_fit = arch_model(
        garch_input,
        mean="Zero",
        dist="normal",
        rescale=False,
        **model_kwargs,
    ).fit(disp="off")
    annualized_model_vol = (model_fit.conditional_volatility / 100.0) * np.sqrt(252)
    annualized_model_vol.name = f"Annualized {model_name}"
    annualized_model_vols[model_name] = annualized_model_vol

def compute_rolling_realized_vol_series(window, method):
    if method == "close-to-close":
        return returns.rolling(window).std() * np.sqrt(252)

    volatility_frame = rolling.volatility(ohlc_frame, windows=(window,), method=method)
    return volatility_frame.iloc[:, 0]

def compute_ewma_realized_vol_series(window, method):
    alpha = 2.0 / (window + 1.0)

    if method == "close-to-close":
        ewma_variance = returns.pow(2).ewm(alpha=alpha, adjust=False, min_periods=window).mean()
        return np.sqrt(ewma_variance * 252)

    if method == "parkinson":
        ewma_variance = parkinson_variance.ewm(alpha=alpha, adjust=False, min_periods=window).mean().clip(lower=0)
        return np.sqrt(ewma_variance * 252)

    if method == "garman-klass":
        ewma_variance = garman_klass_variance.ewm(alpha=alpha, adjust=False, min_periods=window).mean().clip(lower=0)
        return np.sqrt(ewma_variance * 252)

    if method == "rogers-satchell":
        ewma_variance = rs_variance.ewm(alpha=alpha, adjust=False, min_periods=window).mean().clip(lower=0)
        return np.sqrt(ewma_variance * 252)

    if method == "yang-zhang":
        if window < 2:
            return pd.Series(np.nan, index=ohlc_frame.index)

        k = 0.34 / (1.34 + ((window + 1) / (window - 1)))
        overnight_variance = log_oc.ewm(alpha=alpha, adjust=False, min_periods=window).var(bias=False)
        open_to_close_variance = log_co.ewm(alpha=alpha, adjust=False, min_periods=window).var(bias=False)
        rs_component = rs_variance.ewm(alpha=alpha, adjust=False, min_periods=window).mean()
        yz_variance = (
            overnight_variance
            + (k * open_to_close_variance)
            + ((1 - k) * rs_component)
        ).clip(lower=0)
        return np.sqrt(yz_variance * 252)

    raise ValueError(f"Unsupported EWMA volatility method: {method}")

def smooth_annualized_model_vol(model_vol, window):
    # Smooth model variance over the same horizon used by the trailing realized-vol estimator.
    return np.sqrt(model_vol.pow(2).rolling(window).mean())

volatility_term_order = [term for term in time_frame_map if time_frame_map.get(term) is not None]
default_vol_term = 'long' if 'long' in volatility_term_order else max(
    volatility_term_order,
    key=lambda term: int(time_frame_map[term]),
)

vol_model_fig = make_subplots(
    rows=4,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.06,
    subplot_titles=(
        "Annualized GARCH Family vs Close-to-Close Volatility (Raw + Smoothed)",
        "Annualized Rolling and EWMA Volatility Estimators",
        "Spread vs Close-to-Close Annualized Volatility",
        "Annualized Rolling Minus EWMA Volatility Estimators",
    ),
)
term_trace_bounds = {}
term_ranges = {}

for term in volatility_term_order:
    window = int(time_frame_map[term])
    rolling_realized_vol_map = {
        label: compute_rolling_realized_vol_series(window, method)
        for label, method, _, _ in rolling_realized_vol_specs
    }
    ewma_realized_vol_map = {
        label: compute_ewma_realized_vol_series(window, method)
        for label, method, _, _ in ewma_realized_vol_specs
    }
    baseline_vol = rolling_realized_vol_map["Close-to-Close"]

    visible = term == default_vol_term
    term_trace_start = len(vol_model_fig.data)

    vol_model_fig.add_trace(
        go.Scatter(
            x=baseline_vol.index,
            y=baseline_vol,
            mode="lines",
            name=f"Close-to-Close ({window}-day)",
            line=dict(color="#1f77b4", width=2),
            visible=visible,
            showlegend=visible,
            legendgroup="close-to-close-baseline",
        ),
        row=1,
        col=1,
    )

    for model_name, _, color, dash in volatility_model_specs:
        model_vol = annualized_model_vols[model_name]
        smoothed_model_vol = smooth_annualized_model_vol(model_vol, window)
        model_spread = (model_vol - baseline_vol).dropna()
        smoothed_model_spread = (smoothed_model_vol - baseline_vol).dropna()

        vol_model_fig.add_trace(
            go.Scatter(
                x=model_vol.index,
                y=model_vol,
                mode="lines",
                name=f"Annualized {model_name}",
                line=dict(color=color, width=2, dash=dash),
                visible=visible,
                showlegend=visible,
                legendgroup=model_name,
            ),
            row=1,
            col=1,
        )

        vol_model_fig.add_trace(
            go.Scatter(
                x=smoothed_model_vol.index,
                y=smoothed_model_vol,
                mode="lines",
                name=f"{model_name} Smoothed ({window}-day)",
                line=dict(color=color, width=3, dash="longdash"),
                opacity=0.9,
                visible=visible,
                showlegend=visible,
                legendgroup=f"{model_name}-smoothed",
            ),
            row=1,
            col=1,
        )

        vol_model_fig.add_trace(
            go.Scatter(
                x=model_spread.index,
                y=model_spread,
                mode="lines",
                name=f"{model_name} Spread",
                line=dict(color=color, width=2, dash=dash),
                visible=visible,
                showlegend=False,
                legendgroup=f"{model_name}-spread",
            ),
            row=3,
            col=1,
        )

        vol_model_fig.add_trace(
            go.Scatter(
                x=smoothed_model_spread.index,
                y=smoothed_model_spread,
                mode="lines",
                name=f"{model_name} Smoothed Spread",
                line=dict(color=color, width=3, dash="longdash"),
                opacity=0.9,
                visible=visible,
                showlegend=False,
                legendgroup=f"{model_name}-smoothed-spread",
            ),
            row=3,
            col=1,
        )

    for label, _, color, dash in rolling_realized_vol_specs:
        realized_vol = rolling_realized_vol_map[label]

        vol_model_fig.add_trace(
            go.Scatter(
                x=realized_vol.index,
                y=realized_vol,
                mode="lines",
                name=f"{label} ({window}-day)",
                line=dict(color=color, width=2, dash=dash),
                visible=visible,
                showlegend=False,
                legendgroup=f"rolling-realized-{label}",
            ),
            row=2,
            col=1,
        )

        if label != "Close-to-Close":
            realized_spread = (realized_vol - baseline_vol).dropna()
            vol_model_fig.add_trace(
                go.Scatter(
                    x=realized_spread.index,
                    y=realized_spread,
                    mode="lines",
                    name=f"{label} Spread",
                    line=dict(color=color, width=2, dash=dash),
                    visible=visible,
                    showlegend=False,
                    legendgroup=f"rolling-realized-spread-{label}",
                ),
                row=3,
                col=1,
            )

    for label, method, color, dash in ewma_realized_vol_specs:
        ewma_vol = ewma_realized_vol_map[label]

        vol_model_fig.add_trace(
            go.Scatter(
                x=ewma_vol.index,
                y=ewma_vol,
                mode="lines",
                name=f"{label} ({window}-day)",
                line=dict(color=color, width=2, dash=dash),
                opacity=0.9,
                visible=visible,
                showlegend=False,
                legendgroup=f"ewma-realized-{label}",
            ),
            row=2,
            col=1,
        )

        ewma_spread = (ewma_vol - baseline_vol).dropna()
        vol_model_fig.add_trace(
            go.Scatter(
                x=ewma_spread.index,
                y=ewma_spread,
                mode="lines",
                name=f"{label} Spread",
                line=dict(color=color, width=2, dash=dash),
                opacity=0.9,
                visible=visible,
                showlegend=False,
                legendgroup=f"ewma-realized-spread-{label}",
            ),
            row=3,
            col=1,
        )

    for (rolling_label, _, color, _), (ewma_label, _, _, _) in zip(rolling_realized_vol_specs, ewma_realized_vol_specs):
        realized_minus_ewma = (rolling_realized_vol_map[rolling_label] - ewma_realized_vol_map[ewma_label]).dropna()

        vol_model_fig.add_trace(
            go.Scatter(
                x=realized_minus_ewma.index,
                y=realized_minus_ewma,
                mode="lines",
                name=f"{rolling_label} Minus {ewma_label} ({window}-day)",
                line=dict(color=color, width=3),
                opacity=0.9,
                visible=visible,
                showlegend=False,
                legendgroup=f"realized-minus-ewma-{rolling_label}",
            ),
            row=4,
            col=1,
        )

    term_trace_bounds[term] = (term_trace_start, len(vol_model_fig.data))

    non_empty_series = [
        series
        for series in (
            [series.dropna() for series in rolling_realized_vol_map.values()]
            + [series.dropna() for series in ewma_realized_vol_map.values()]
            + [model_series.dropna() for model_series in annualized_model_vols.values()]
        )
        if not series.empty
    ]
    if non_empty_series:
        max_index = max(series.index.max() for series in non_empty_series)
        min_index = min(series.index.min() for series in non_empty_series)
        term_ranges[term] = [max(min_index, max_index - pd.DateOffset(years=3)), max_index]
    else:
        term_ranges[term] = None

vol_model_fig.add_hline(
    y=0,
    line_dash="dot",
    line_color="rgba(120, 120, 120, 0.8)",
    row=3,
    col=1,
)
vol_model_fig.add_hline(
    y=0,
    line_dash="dot",
    line_color="rgba(120, 120, 120, 0.8)",
    row=4,
    col=1,
)

vol_buttons = []
total_traces = len(vol_model_fig.data)
for term in volatility_term_order:
    visibility = [False] * total_traces
    start, end = term_trace_bounds[term]
    for trace_idx in range(start, end):
        visibility[trace_idx] = True

    layout_updates = {
        "title": {
            "text": f"{ticker_str} Volatility Models and Estimators ({time_frame_map[term]}-Day)",
            "x": 0.5,
            "xanchor": "center",
            "y": 0.97,
            "yanchor": "top",
        },
        "yaxis": {"title": "Annualized Volatility"},
        "yaxis2": {"title": "Annualized Volatility"},
        "yaxis3": {"title": "Spread vs Close-to-Close"},
        "yaxis4": {"title": "Rolling Minus EWMA"},
    }
    if term_ranges.get(term) is not None:
        layout_updates["xaxis"] = {"range": term_ranges[term]}
        layout_updates["xaxis2"] = {"range": term_ranges[term]}
        layout_updates["xaxis3"] = {"range": term_ranges[term]}
        layout_updates["xaxis4"] = {"range": term_ranges[term]}

    vol_buttons.append(
        dict(
            label=f"{term.title()} ({time_frame_map[term]})",
            method="update",
            args=[{"visible": visibility}, layout_updates],
        )
    )

available_ranges = [date_range for date_range in term_ranges.values() if date_range is not None]
if available_ranges:
    global_start = min(date_range[0] for date_range in available_ranges)
    global_end = max(date_range[1] for date_range in available_ranges)
    default_range = term_ranges[default_vol_term] or [global_start, global_end]
    vol_model_fig.update_xaxes(range=default_range, row=1, col=1)
    vol_model_fig.update_xaxes(range=default_range, row=2, col=1)
    vol_model_fig.update_xaxes(range=default_range, row=3, col=1)
    vol_model_fig.update_xaxes(range=default_range, row=4, col=1)
    time_range_menu = dict(
        type="dropdown",
        buttons=build_time_range_buttons(global_start, global_end, axis_count=4),
        x=0.28,
        xanchor="left",
        y=1.08,
        yanchor="top",
        showactive=True,
    )
else:
    time_range_menu = None

vol_model_fig.update_layout(
    template="plotly_white",
    height=1450,
    margin=dict(t=150),
    legend=dict(x=0.01, y=0.99),
    yaxis_title="Annualized Volatility",
    yaxis2_title="Annualized Volatility",
    yaxis3_title="Spread vs Close-to-Close",
    yaxis4_title="Rolling Minus EWMA",
    title=dict(
        text=f"{ticker_str} Volatility Models and Estimators ({time_frame_map[default_vol_term]}-Day)",
        x=0.5,
        xanchor="center",
        y=0.97,
        yanchor="top",
    ),
    updatemenus=[
        dict(
            type="dropdown",
            buttons=vol_buttons,
            x=0.0,
            xanchor="left",
            y=1.08,
            yanchor="top",
            showactive=True,
            active=volatility_term_order.index(default_vol_term),
        )
    ] + ([time_range_menu] if time_range_menu is not None else []),
)
show_plotly_figure(vol_model_fig)